# Mean Meridional Velocity at 26.5°N — MOM6 Transect

Computes the time-mean meridional velocity (`vo`) from MOM6 z-level output at the 26.5°N DWBC transect (77°W–70°W), saves it to a netCDF file, and produces a figure comparable to the observational transects in Biló & Johns (2020).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%%capture 
# comment above line to see details about the run(s) displayed
from misc import *
from scipy.interpolate import interp1d
import xarray as xr
import matplotlib as mpl
%matplotlib inline

In [ ]:
# Observational / topography data paths
DWBC_DATA = '/glade/work/gmarques/cesm/datasets/DWBC'

## Read netCDF data

In [ ]:
ds = []
for c, p in zip(casename, ocn_path):
  ds = xr.open_dataset(p+'{}_vo_mean_26.5N_transect.nc'.format(c))

In [ ]:
vo_mean = ds['vo']
deptho  = ds['deptho']

lat_transect = ds.attrs['latitude']
lon_min     = ds.attrs['lon_min']
lon_max     = ds.attrs['lon_max']
start_date  = ds.attrs['start_date']
end_date    = ds.attrs['end_date']

## Helper: build base transect figure (following Biló & Johns 2020 style)

In [ ]:
def base_26N_transect(lon_min=-77.0, lon_max=-70.0):
    """Return (fig, ax_lon, ax_dist) with topography and station markers."""
    # Observed topography from echo-sounder surveys
    topography = xr.open_dataset(DWBC_DATA + '/topography_26.5N.nc')
    ftopo = interp1d(topography.longitude.values,
                     topography.topography.values, kind='linear')

    stations = xr.open_dataset(DWBC_DATA + '/stations_positions.nc')
    moorings = xr.open_dataset(DWBC_DATA + '/mooring_instrumentation.nc')
    mtopo    = ftopo(moorings.longitude.values)

    fig    = plt.figure(figsize=(10, 7), facecolor='w')
    ax_lon = plt.gca()
    ax_dist = ax_lon.twiny()

    # Stations
    ax_lon.plot(stations.longitude.values,
                np.zeros(stations.longitude.shape[0]),
                'o', mfc='darkorange', markeredgecolor='k', markersize=7, zorder=9)

    # Topography fill
    ax_lon.fill_between(topography.longitude.values, topography.topography,
                        y2=7000, color='k', zorder=10)

    # Mooring positions
    ax_lon.plot(moorings.longitude, mtopo + 120.0,
                '^', mfc='darkorange', markeredgecolor='k', ms=12, zorder=11)

    # Axes layout
    ax_lon.xaxis.tick_bottom()
    ax_dist.xaxis.tick_top()
    ax_dist.set_xlabel('Distance (km)', fontsize=18)
    ax_lon.set_ylabel('Depth (m)', fontsize=22)

    lon_range     = np.arange(-77.0, -69.0, 1.0)
    lon_range_str = [r'%i$^{\circ}$W' % int(np.abs(l)) for l in lon_range]

    ax_lon.set_yticks(range(0, 5500, 500))
    ax_lon.set_xticks(lon_range)
    ax_lon.set_xticklabels(lon_range_str, color='k', fontsize=18)
    ax_dist.set_xticks(range(0, 700, 50))
    ax_dist.tick_params(labelsize=15)
    ax_lon.tick_params(labelsize=18)

    ax_lon.set_ylim(5250, -20)
    ax_lon.set_xlim(lon_min, lon_max)
    ax_dist.set_xlim(0.0, stations.distance.max())

    return fig, ax_lon, ax_dist

## Plot: model mean meridional velocity transect

In [ ]:
# ---- colour map matching the observation panels ----
boundaries     = np.linspace(-0.3, 0.3, 50)
cmap_seismic   = plt.cm.get_cmap('seismic', len(boundaries))
colors         = list(cmap_seismic(np.arange(len(boundaries))))
cmap_custom    = mpl.colors.ListedColormap(colors, '')
cmap_custom.set_over(colors[-1])
cmap_custom.set_under(colors[0])

kw_color   = dict(vmin=-0.3, vmax=0.3, cmap=cmap_custom, zorder=0)
kw_contour = dict(levels=[-0.15, -0.1, -0.05, 0.03, 0.1, 0.2, 0.3],
                  colors='k', linewidths=1, linestyles='solid', zorder=1)
kw_contourz = dict(levels=[0.0],
                   colors='k', linewidths=2.5, linestyles='solid', zorder=1)

# ---- base figure ----
fig, ax_lon, ax_dist = base_26N_transect(lon_min=lon_min, lon_max=lon_max)

# ---- meridional velocity (pcolormesh + contours) ----
xh  = vo_mean.xh.values
z_l = vo_mean.z_l.values
vel = vo_mean.values   # (z_l, xh)

cm = ax_lon.pcolormesh(xh, z_l, vel, **kw_color)

# Contours only where we have enough data (skip near-surface noise)
try:
    c  = ax_lon.contour(xh, z_l, vel, **kw_contour)
    cz = ax_lon.contour(xh, z_l, vel, **kw_contourz)
    ax_lon.clabel(c,  fmt='%1.2f', colors='k',     fontsize=13, inline=True, zorder=14)
    ax_lon.clabel(cz, fmt='%i',    colors='beige',  fontsize=15, inline=True, zorder=14)
    [label.set_bbox(dict(edgecolor='none', facecolor='k', pad=0))
     for label in ax_lon.findobj(mpl.text.Text) if label.get_text() == '0']
except Exception:
    pass   # skip contour labels if grid is too coarse

# ---- model bathymetry overlay ----
ax_lon.fill_between(deptho.xh.values, deptho.values, y2=7000,
                    color='saddlebrown', alpha=0.6, zorder=8, label='Model bathymetry')

# ---- colour bar ----
cbaxes = fig.add_axes([0.925, 0.115, 0.010, 0.77])
cb = fig.colorbar(cm, cax=cbaxes, extend='both',
                  ticks=np.arange(-0.3, 0.35, 0.05))
cb.ax.tick_params(labelsize=13)
cb.ax.set_title(r'm s$^{-1}$', fontsize=13, x=1.3)

# ---- title ----
ax_lon.text(
    lon_min + 0.3, 500,
    '{:.1f}N  MOM6\n{} – {}'.format(lat_transect, start_date, end_date),
    color='beige', fontsize=14,
    bbox=dict(boxstyle='round', facecolor='k'), zorder=12)

#savefig = '../PNG/DWBC/{}_vo_mean_{:.1f}N_transect.png'.format(casename, lat_transect)
#fig.savefig(savefig, bbox_inches='tight', pad_inches=0.05, dpi=150)
#print('Figure saved:', savefig)
#plt.show()

## Optional: 4-panel comparison (model + 3 observational products)

Reproduces the layout of Fig. 3 in Biló & Johns (2020) with the model panel appended.

In [ ]:
# Load observational datasets
ladcp  = xr.open_dataset(DWBC_DATA + '/ladcp_statistics.nc')
mocha  = xr.open_dataset(DWBC_DATA + '/mocha_currentmeter_dwbc_2008_2018.nc')
argo   = xr.open_dataset(DWBC_DATA + '/argo_vgeos_26.5N.nc')
import gsw
argo_depth = -gsw.z_from_p(argo.pressure.values, lat_transect)

datasets = [
    dict(label='LADCP',
         x=ladcp.longitude.values,  y=ladcp.depth.values,
         v=ladcp.v_mean.values / 100.0,   kind='contourf'),
    dict(label='MOCHA moorings',
         x=mocha.longitude.values, y=mocha.depth.values,
         v=np.nanmean(mocha.v.values, axis=-1).T, kind='contourf'),
    dict(label='Argo geostrophy',
         x=argo.longitude[1:-1].values, y=argo_depth,
         v=argo.v[:, 1:-1].values,      kind='pcolormesh'),
    dict(label='MOM6\n{} – {}'.format(start_date, end_date),
         x=xh, y=z_l,
         v=vel,                          kind='pcolormesh'),
]

topography = xr.open_dataset(DWBC_DATA + '/topography_26.5N.nc')
stations   = xr.open_dataset(DWBC_DATA + '/stations_positions.nc')

fig, axes = plt.subplots(2, 2, figsize=(18, 12), facecolor='w',
                         sharex=False, sharey=True)
axes = axes.flatten()

for ax, d in zip(axes, datasets):
    if d['kind'] == 'contourf':
        cm = ax.contourf(d['x'], d['y'], d['v'],
                         np.linspace(-0.3, 0.3, 50), **kw_color)
        try:
            ax.contour(d['x'], d['y'], d['v'], **kw_contour)
            ax.contour(d['x'], d['y'], d['v'], **kw_contourz)
        except Exception:
            pass
    else:
        cm = ax.pcolormesh(d['x'], d['y'], d['v'], **kw_color)
        try:
            ax.contour(d['x'], d['y'], d['v'], **kw_contour)
            ax.contour(d['x'], d['y'], d['v'], **kw_contourz)
        except Exception:
            pass

    # Topography
    ax.fill_between(topography.longitude.values, topography.topography.values,
                    y2=7000, color='k', zorder=10)

    # Label
    ax.text(0.03, 0.07, d['label'], transform=ax.transAxes,
            color='beige', fontsize=14,
            bbox=dict(boxstyle='round', facecolor='k'), zorder=12)

    ax.set_ylim(5250, -20)
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylabel('Depth (m)', fontsize=14)
    ax.set_xlabel('Longitude', fontsize=14)
    ax.tick_params(labelsize=12)

    lon_ticks = np.arange(-77.0, -69.0, 1.0)
    ax.set_xticks(lon_ticks)
    ax.set_xticklabels([r'%i$^{\circ}$W' % int(abs(l)) for l in lon_ticks], fontsize=11)

# Shared colour bar
fig.subplots_adjust(right=0.88, hspace=0.3, wspace=0.25)
cbaxes = fig.add_axes([0.91, 0.12, 0.012, 0.76])
cb = fig.colorbar(cm, cax=cbaxes, extend='both',
                  ticks=np.arange(-0.3, 0.35, 0.05))
cb.ax.tick_params(labelsize=12)
cb.ax.set_title(r'm s$^{-1}$', fontsize=12, x=1.5)

fig.suptitle('Mean meridional velocity at {:.1f}°N'.format(lat_transect), fontsize=16)

#savefig4 = '../PNG/DWBC/{}_vo_mean_{:.1f}N_4panel.png'.format(casename, lat_transect)
#fig.savefig(savefig4, bbox_inches='tight', pad_inches=0.05, dpi=150)
#print('Figure saved:', savefig4)
#plt.show()